# Notebook 02 — Exploratory Data Analysis and Data Quality

**Project:** Next Day Wildfire Spread Prediction via Tabular ML and Spatial Deep Learning  
**Team:** TensorTitans, Section C  
**Objective:** Explore multi-channel spatial patch structures, visualize wildfire masks alongside environmental drivers, evaluate feature distributions, check correlation structure, and quantify the contrast between burned and unburned terrain.

---
## 1. Environment and Data Setup


In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import src.data_loader as dl

fig_dir = "results/figures" if os.path.exists("results/figures") else "../results/figures"
os.makedirs(fig_dir, exist_ok=True)

# Extract a representative sample of patches
train_paths = dl.get_tfrecord_paths("train")
patches = list(dl.iter_records(train_paths, max_records=15))
print(f"Loaded {len(patches)} representative multi-channel wildfire patches.")


Loaded 15 representative multi-channel wildfire patches.


I0000 00:00:1792988639.134082 2270336 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


## 2. Multi-Channel Spatial Patch and Wildfire Mask Visualizations
Wildfire propagation is governed by interacting physical processes: antecedent fire perimeter (`PrevFireMask`), topography (`elevation`), weather drivers (`tmmx`, `vs`), and fuel dryness (`erc`).
Below we visualize a 6-channel spatial slice for an active wildfire event.


In [ ]:
active_patch = None
for p in patches:
    if np.sum(p["FireMask"] == 1.0) > 10 and np.sum(p["PrevFireMask"] == 1.0) > 10:
        active_patch = p
        break

if active_patch is None:
    active_patch = patches[0]

channels_to_plot = [
    ("PrevFireMask", "Day t Active Fire (PrevFireMask)", "Reds"),
    ("FireMask", "Day t+1 Target Fire (FireMask)", "OrRd"),
    ("tmmx", "Max Temperature (K)", "magma"),
    ("vs", "Wind Speed 10m (m/s)", "viridis"),
    ("elevation", "Elevation (m)", "terrain"),
    ("erc", "Energy Release Component", "inferno")
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for idx, (ch, title, cmap) in enumerate(channels_to_plot):
    ax = axes[idx // 3, idx % 3]
    im = ax.imshow(active_patch[ch], cmap=cmap)
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axis("off")

plt.suptitle("Multi-Channel Spatial Representation of Wildfire Spread Event (64x64 km)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "multichannel_spatial_patch.png"), dpi=250)
plt.show()

## 3. Tabular Extraction and Environmental Feature Distributions
We extract pixel-level tabular observations from the audited patches to evaluate distributions and check for anomalies or missing values.


In [ ]:
tabular_df = dl.extract_split_tabular("train", max_patches=15, sample_negatives_ratio=10.0)
print(f"Extracted {len(tabular_df)} pixel records across 15 patches.")

features_to_hist = ["tmmx", "tmmn", "sph", "vs", "elevation", "erc", "NDVI", "pdsi"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for idx, col in enumerate(features_to_hist):
    sns.histplot(tabular_df[col], kde=True, ax=axes[idx], color="teal", bins=30)
    axes[idx].set_title(f"Distribution of {col}", fontsize=11, fontweight="bold")
    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Environmental Feature Distributions Across Wildfire Samples", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "feature_distributions.png"), dpi=250)
plt.show()